# Tagalog ⇄ Cuyonon — train on Colab

QLoRA fine-tune of `Qwen/Qwen2.5-1.5B-Instruct`, every language direction in
`languages.json`, on a free **T4**.

**Before you start:**
- `Runtime → Change runtime type → T4 GPU → Save`. New notebooks default to a CPU
  runtime and training will not work on CPU.
- This notebook clones the repository from GitHub, so **commit and push** your
  local changes first — otherwise Colab trains on the old code and data.

Run the cells top to bottom. No kernel restart is needed — training/eval run as
subprocesses, so they pick up freshly installed packages.

## 1. Confirm the GPU

In [ ]:
import subprocess, torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → T4 GPU, then re-run this cell."
)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

## 2. Get the code

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("translator"):
    !git clone https://github.com/jubilcabrestante/translator.git
os.chdir("/content/translator")
!git pull --ff-only
print("cwd:", os.getcwd())

## 3. Install dependencies

Pip prints dependency-conflict warnings about `gradio` / `google-adk` / `gcsfs` —
those packages are unrelated to training and the warnings are safe to ignore.

`bitsandbytes` is deliberately **not** version-pinned in `requirements.txt`: the
wheel has to match the runtime's CUDA and Triton. Pinning 0.44.1 breaks Colab
(no CUDA-12.8 binary, and it imports `triton.ops`, removed in Triton 3.1).

In [ ]:
!pip install -q -r requirements.txt

# Verify bitsandbytes actually imports AND found its CUDA binary before training.
import bitsandbytes, importlib.metadata as md
print("bitsandbytes", md.version("bitsandbytes"), "imported OK")

## 4. Where to write checkpoints

Colab wipes the VM disk on disconnect (idle ~90 min, hard cap ~12 h). Mount Drive
so a checkpoint survives and the training cell can resume. Skip the mount and
`OUT_DIR` stays on the ephemeral disk — fine for a single uninterrupted run.

In [ ]:
OUT_DIR = "outputs/translator-lora"  # ephemeral VM disk

# --- To persist to Drive instead, uncomment these three lines: ---
# from google.colab import drive
# drive.mount("/content/drive")
# OUT_DIR = "/content/drive/MyDrive/translator/outputs/translator-lora"

print("checkpoints →", OUT_DIR)

## 5. Check and build the dataset

`lint_data.py` reports data problems (it changes nothing without `--fix`).
`prepare_data.py` reads the corpora in `languages.json` plus `assets/entities/`
and writes `data/train.jsonl`, `data/val.jsonl` and `data/test.jsonl`.
Check the printed share of Cuyonon in the training mix.

In [ ]:
!python scripts/lint_data.py --show 3
!python scripts/prepare_data.py

## 6. Train

Three epochs over the current data takes on the order of an hour or more on a
T4 (longer with reference corpora downloaded). The epoch with the lowest
validation loss is the one saved, and the loss is computed on the answer only
(add `--full-sequence-loss` to compare with the old behaviour). If `OUT_DIR`
already holds a checkpoint (e.g. after a disconnect) this resumes from the latest
one.

In [ ]:
import glob
cks = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"),
             key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = f"--resume-from-checkpoint {cks[-1]}" if cks else ""
print("resume:", resume or "(fresh run)")
!python scripts/train.py --output-dir {OUT_DIR} {resume}

## 7. Evaluate (chrF with 95% intervals, BLEU, exact-match)

Scores the frozen test set if `assets/test-taga-cuyo.txt` has rows, otherwise the
validation split. Saves predictions and a 50-sentence **review sheet** for a
native speaker under `outputs/eval/`. Add `--limit 200` for a quicker look.

In [ ]:
!python scripts/evaluate.py --adapter-dir {OUT_DIR}

Download the review sheet, have a Cuyonon speaker fill in the *rating* column
(correct / understandable / wrong), then on your machine run
`python scripts/evaluate.py --tally review-sheet.csv`.

In [ ]:
import glob
from google.colab import files
latest = sorted(glob.glob("outputs/eval/*/review-sheet.csv"))[-1]
print("review sheet:", latest)
files.download(latest)

## 8. Spot-check a few translations

In [ ]:
!python scripts/translate.py --adapter-dir {OUT_DIR} --text "Magandang umaga sa inyong lahat." --to Cuyonon
!python scripts/translate.py --adapter-dir {OUT_DIR} --text "Saan ka pupunta?" --to Cuyonon

## 9. Save the adapter off the VM

If you mounted Drive in step 4 it is already saved there. Otherwise zip just the
adapter (not the multi-hundred-MB optimizer checkpoints) and download it.

In [ ]:
!cd "{OUT_DIR}" && zip -qr /content/translator-lora.zip . -x 'checkpoint-*/*'
from google.colab import files
files.download("/content/translator-lora.zip")

## 10. (Optional) Export for cheap CPU hosting

Merges the adapter into the base model and converts it to GGUF, which Ollama or
llama.cpp run on an ordinary CPU server; `scripts/serve.py` can then forward to it
(backend mode, see the README). Run this last: the converter's requirements may
change installed package versions.

In [ ]:
!python scripts/export_model.py --adapter-dir {OUT_DIR} --out-dir outputs/translator-merged
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /content/llama.cpp/convert_hf_to_gguf.py outputs/translator-merged --outtype q8_0 --outfile outputs/translator-merged/translator-q8_0.gguf
from google.colab import files
files.download("outputs/translator-merged/translator-q8_0.gguf")
files.download("outputs/translator-merged/Modelfile")